# Intro

This notebook is used to analyze latent features (including attention maps, FFN outputs, etc.) of **StreamVGGT**.

**Note:** StreamVGGT is a causal streaming model that differs from VGGT, $\pi^3$, and DA3 in that, when processing the $i$-th frame, it can only access the current frame and the previously processed frames (i.e., $0$ to $i$).

In [ ]:
import os
import shutil
import glob
import math
import random
import argparse
from addict import Dict
from collections import defaultdict

from IPython.display import Image, display, HTML
import pycolmap
import open3d as o3d
import cv2
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
import torch
import torch.nn as nn
import torch.nn.functional as F

# Utils
from saf3r.utils.model_utils import build_model, infer_model
from saf3r.utils.probe_utils import *
from saf3r.utils.vis_utils import *


# Unified model device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# NOTE: Automatically remove previously saved Plotly figures at the start of each run
if os.path.exists("iframe_figures"):
    shutil.rmtree("iframe_figures")

## Load a single scene to inspect

**Note:** Since we materialize and store the full attention maps later, using too many frames may result in an out-of-memory error. We can select a small subset of frames using `frame_ids` below.

In [ ]:
# Set data directory
scene_dir = "../data/7s_chess_seq01"

# Set the indices of the frames to inspect
# NOTE: Since we materialize the full attention maps later, using too many frames may result in OOM
frame_ids = [0, 1, 5, 8]

# Fetch all image paths
image_files = sorted(glob.glob(os.path.join(scene_dir, "*.png")))  # Assume input images are PNG files

# Select a subset based on frame_ids
scene_data = Dict()
scene_data.image_files = [image_files[i] for i in frame_ids]
print("Selected frames:\n", scene_data.image_files)

## Load model

In [ ]:
model_cfg = Dict(
    name="StreamVGGT", ckpt_path="../checkpoints/streamvggt/checkpoints.pth"
)
model = build_model(model_cfg, device)

## Analyze attention maps

### ⇒ Forward the model to obtain internal states

In [ ]:
# Register hooks
# NOTE: AA hooks are always enabled
frame_attn_maps, global_attn_maps = defaultdict(list), defaultdict(list)
g_q, g_k, g_v = defaultdict(list), defaultdict(list), defaultdict(list)
g_q, g_k, g_v = None, None, None
frame_attn_hooks, global_attn_hooks = register_alt_attn_hooks(model, "streamvggt", frame_attn_maps, global_attn_maps, g_q, g_k, g_v)

# Forward and get attention maps
pred_data, stats = infer_model(model, model_cfg, scene_data)

# Remove hooks
remove_hooks(frame_attn_hooks, global_attn_hooks)


# Collect meta info
PATCH_SIZE = 14
NUM_SPECIAL_TOKENS = 5
imgs = np.split(pred_data.processed_images, pred_data.processed_images.shape[0]) # [S, H, W, 3]
num_imgs = len(imgs)
num_frame_layers, num_global_layers = len(frame_attn_maps[0]), len(global_attn_maps[0])
num_heads = global_attn_maps[0][0].shape[1]
grid_h, grid_w = imgs[0].shape[1]//PATCH_SIZE, imgs[0].shape[2]//PATCH_SIZE
tokens_per_imgs = NUM_SPECIAL_TOKENS + grid_h * grid_w
print(f"# of Local Layers: {num_frame_layers}, # of Global Layers: {num_global_layers}")
print(f"# of Special Token: {NUM_SPECIAL_TOKENS}, # of Image Tokens: {grid_h*grid_w} ({grid_h} x {grid_w})")
print(f"# of Attention Heads: {num_heads}")


# NOTE: The following code is specific to streaming models
# Merge results from each frame to a single attention map of shape [num_global_layers, num_heads, N, N],
# where N is the total number of tokens from all frames
tot_len = global_attn_maps[num_imgs - 1][0].shape[-1]

# pad the first N-1 images
pad_global_attn_maps = []
for i in range(num_global_layers):
    pad_attn_maps = []
    for j in range(num_imgs):
        cur_len = global_attn_maps[j][i].shape[-1]
        pad_attn_map = F.pad(global_attn_maps[j][i], (0, tot_len - cur_len))
        pad_attn_maps.append(pad_attn_map)
    pad_attn_maps = torch.cat(pad_attn_maps, dim=2)  # [1, 16, N, N]
    pad_global_attn_maps.append(pad_attn_maps)

global_attn_maps = pad_global_attn_maps

### ⇒ Analyze alternating attention (Frame/Global)

#### Inspecting the attention distribution for a single query

We provide different options for inspecting different attention patterns.

**Supported args:**

- `attn_type`: Specifies the attention type to visualize. Options: `global` (Only global attention is currently supported for StreamVGGT).
- `layer_idx`: Index of the attention layer to visualize.
- `head`: Attention head index. Can be an integer or `mean` to average over all heads.
- `query_type`: Specifies the query token type (special tokens or image tokens). Options: `special`, `img`.
- `query_idx`: Query token index.
  - For image tokens, use `(img_idx, row_idx, col_idx)`.
  - For special tokens, use `(img_idx, token_idx)`.
  - Set to `None` to randomly select a query token.

In [ ]:
# ================================================== #
# MANUALLY define which query token to visualize
# ================================================== #
attn_type = "global"
layer_idx = 14
head = 4
query_type = "img"
query_idx = (1, 23, 25)


# ================================================== #
# AUTO parser
# ================================================== #
assert attn_type == "global"
if query_type == "img":
    if query_idx is None:
        query_idx = (random.randint(0, num_imgs-1), random.randint(0, grid_h-1), random.randint(0, grid_w-1))
    assert query_idx[0] <= num_imgs and query_idx[1] <= grid_h and query_idx[2] <= grid_w
    query_img_idx = query_idx[0]
    query_local_idx = NUM_SPECIAL_TOKENS + (query_idx[1]*grid_w + query_idx[2])
    print(f"AA: {attn_type} | Layer: {layer_idx} | Frame: {query_idx[0]} | Patch: ({query_idx[1]}, {query_idx[2]})")
else:
    if query_idx is None:
        query_idx = (random.randint(0, num_imgs-1), random.randint(0, NUM_SPECIAL_TOKENS-1))
    assert  query_idx[0] <= num_imgs and query_idx[1] <= NUM_SPECIAL_TOKENS
    query_img_idx, query_local_idx = query_idx
    print(f"AA: {attn_type} | Layer: {layer_idx} | Frame: {query_idx[0]} | Special token: {query_local_idx}")


# ================================================== #
# Plot attn-img overlay figures with highlights
# ================================================== #
topk_highlight = 15
plot_attn_img_overlay(
    imgs, attn_global=torch.stack(global_attn_maps), attn_local=None, mode=attn_type,
    layer=layer_idx, head=head, query_img_idx=query_img_idx, query_local_idx=query_local_idx,
    grid_h=grid_h, grid_w=grid_w, num_special=NUM_SPECIAL_TOKENS, special_token_arrange="first",
    cmap="cividis",
    overlay_alpha=0.4, query_color="blue", save_path="tmp_plots/streamvggt/attn_img_overlay.png",
    # highlight TopK
    topk=topk_highlight, topk_include_special=True, topk_show_mode="hard", topk_show_rank=True, # draw rank numbers on topk patches
)
display(HTML('<img src="tmp_plots/streamvggt/attn_img_overlay.png">'))


# ================================================== #
# Plot per-query attn stats
# ================================================== #
if attn_type == "global":
    q_global = query_img_idx * tokens_per_imgs + query_local_idx
    attn_probs = global_attn_maps[layer_idx][0, :, q_global, :].float()
else: # local
    q_global = query_local_idx
    attn_probs = frame_attn_maps[layer_idx][query_img_idx, :, q_global, :].float()
attn_probs = attn_probs.mean(0) if head == "mean" else attn_probs[head]
# pdf
plot_attn_distribution(
    attn_probs=attn_probs, cur_idx=q_global, sort_desc=False, mode="pdf",
    tokens_per_imgs=tokens_per_imgs, topk=topk_highlight, save_path="tmp_plots/streamvggt/attn_pdf.png"
)
display(HTML('<img src="tmp_plots/streamvggt/attn_pdf.png">'))
# cdf
plot_attn_distribution(
    attn_probs=attn_probs, sort_desc=True, mode="cdf",
    tokens_per_imgs=tokens_per_imgs, topk=topk_highlight, save_path="tmp_plots/streamvggt/attn_cdf.png"
)
display(HTML('<img src="tmp_plots/streamvggt/attn_cdf.png">'))

#### Plot attention map for the selected head

In [ ]:
if attn_type == "global":
    attn_map = global_attn_maps[layer_idx][0, :, :, :].float().cpu().numpy() # [H, N, N]
else:
    attn_map = frame_attn_maps[layer_idx][query_img_idx, :, :, :].float().cpu().numpy() # [H, N, N]
attn_map_disp = attn_map.mean(0) if head == "mean" else attn_map[head]
plot_full_attn_map(
    attn_map_disp, cmap="magma", tokens_per_imgs=tokens_per_imgs, cur_token=q_global, normalize=True, gamma=0.5,
    subtitle=f"Attention Map ({attn_type}) " + ("AvgHead" if head == "mean" else f"Head {head}") + f" (Layer {layer_idx})",
    save_path="tmp_plots/streamvggt/attn_map.png"
)
display(HTML('<img src="tmp_plots/streamvggt/attn_map.png">'))

#### Plot attention maps of all heads

**Note:** This may take a while

In [ ]:
if attn_type == "global":
    attn_map = global_attn_maps[layer_idx][0, :, :, :].float().cpu().numpy() # [H, N, N]
else:
    attn_map = frame_attn_maps[layer_idx][query_img_idx, :, :, :].float().cpu().numpy() # [H, N, N]
plot_multiheads_full_attn_map(
    attn_map, tokens_per_imgs=tokens_per_imgs, normalize=True, gamma=0.4,
    subtitle=f"Attention Maps ({attn_type}) Across All Heads (Layer {layer_idx})", save_path="tmp_plots/streamvggt/attn_map_heads.png"
)
display(HTML('<img src="tmp_plots/streamvggt/attn_map_heads.png">'))